# Inception Results Inspector

Toggle `MODEL` and `TRIAL_IDX` at the top, then re-run the relevant section cell.

In [7]:
import pandas as pd
import matplotlib.pyplot as plt
import os
%matplotlib inline

# ── Toggle these ───────────────────────────────────────────────────────────────
# Available nicks:
#   'DeepSeek-V4-Pro', 'DeepSeek-V4-Flash', 'Kimi-K2.6', 'GLM-5.1'
#   'DeepSeek-R1-0528', 'DeepSeek-V3.2', 'Qwen3-235B-A22B-Thinking-2507',
#   'Qwen3-Next-80B-A3B-Thinking', 'Kimi-K2-Thinking', 'GLM-4.6', 'GPT-OSS-120B'
MODEL = 'DeepSeek-V4-Flash'
TRIAL_IDX = 0           # set to None for runs without trial subdirs

# Point at results_smoke to inspect Phase 1 smoke data,
# switch to '../results' once Phase 2 fan-out runs complete.
RESULTS_ROOT = os.path.abspath('../results_smoke')

# ── Path helpers ───────────────────────────────────────────────────────────────
# Code writes:
#   benchmark:     RESULTS_ROOT/benchmark/think/[trial_{N}/]{nick}.pickle
#   simple_inject: RESULTS_ROOT/simple_inject/think/[trial_{N}/]{nick}.pickle
#   ablations:     RESULTS_ROOT/think/max_iterations_1/[architect_initial_max_tokens_{T}/][trial_{N}/]{nick}.pickle
#   max_iter_5:    RESULTS_ROOT/think/max_iterations_5/[trial_{N}/]{nick}.pickle

def _p(*parts):
    return os.path.join(*[p for p in parts if p])

def load(variant, model=MODEL, trial_idx=TRIAL_IDX):
    t = f'trial_{trial_idx}' if trial_idx is not None else None
    if variant == 'benchmark':
        path = _p(RESULTS_ROOT, 'benchmark', 'think', t, f'{model}.pickle')
    elif variant == 'simple_inject':
        path = _p(RESULTS_ROOT, 'simple_inject', 'think', t, f'{model}.pickle')
    elif variant == 'max_iter_5':
        path = _p(RESULTS_ROOT, 'think', 'max_iterations_5', t, f'{model}.pickle')
    elif variant == 'ablation_256':
        path = _p(RESULTS_ROOT, 'think', 'max_iterations_1', t, f'{model}.pickle')
    elif variant.startswith('ablation_'):
        tokens = variant.split('_')[1]
        path = _p(RESULTS_ROOT, 'think', 'max_iterations_1',
                  f'architect_initial_max_tokens_{tokens}', t, f'{model}.pickle')
    else:
        raise ValueError(f'Unknown variant: {variant}')
    if not os.path.exists(path):
        print(f'not found: {path}')
        return None
    return pd.read_pickle(path)

def load_all_trials(variant, model=MODEL, n_trials=10):
    frames = []
    for t in range(n_trials):
        df = load(variant, model=model, trial_idx=t)
        if df is not None:
            df = df.copy(); df['trial'] = t; frames.append(df)
    if not frames:
        print('No trials found.'); return None
    out = pd.concat(frames, ignore_index=True)
    print(f'Loaded {len(frames)} trial(s), {len(out)} total rows')
    return out

print(f'MODEL={MODEL}  TRIAL_IDX={TRIAL_IDX}')
print(f'RESULTS_ROOT = {RESULTS_ROOT}')

MODEL=DeepSeek-V4-Flash  TRIAL_IDX=0
RESULTS_ROOT = /home/md2292/inception-frontier-open-weight-models/results_smoke


## 1. Benchmark

In [8]:
df_bench = load('benchmark')
if df_bench is not None:
    print(f'shape: {df_bench.shape}')
    print(f'columns: {list(df_bench.columns)}')
    display(df_bench.head(3))

    fig, axes = plt.subplots(1, 2, figsize=(12, 3))
    think_len = df_bench['thinking'].str.len().dropna() if 'thinking' in df_bench.columns else pd.Series(dtype=float)
    resp_len  = df_bench['response'].str.len().dropna()  if 'response'  in df_bench.columns else pd.Series(dtype=float)
    axes[0].hist(think_len, bins=30, color='steelblue')
    axes[0].set_title(f'{MODEL} benchmark — thinking length'); axes[0].set_xlabel('chars')
    axes[1].hist(resp_len, bins=30, color='coral')
    axes[1].set_title(f'{MODEL} benchmark — response length'); axes[1].set_xlabel('chars')
    plt.tight_layout(); plt.show()

    if 'error' in df_bench.columns:
        n_err = df_bench['error'].notna().sum()
        print(f'rows with error: {n_err}/{len(df_bench)}')

not found: /home/md2292/inception-frontier-open-weight-models/results_smoke/benchmark/think/trial_0/DeepSeek-V4-Flash.pickle


## 2. Simple Inject

In [3]:
df_si = load('simple_inject')
if df_si is not None:
    print(f'shape: {df_si.shape}')
    print(f'columns: {list(df_si.columns)}')
    display(df_si.head(3))

    if 'injection_prefix' in df_si.columns:
        print(f'injection_prefix populated: {df_si["injection_prefix"].notna().all()}')
        print(f'prefix value: {repr(df_si["injection_prefix"].iloc[0])}')

    resp_len = df_si['response'].str.len().dropna() if 'response' in df_si.columns else pd.Series(dtype=float)
    plt.figure(figsize=(6, 3))
    plt.hist(resp_len, bins=30, color='mediumseagreen')
    plt.title(f'{MODEL} simple_inject — response length'); plt.xlabel('chars')
    plt.tight_layout(); plt.show()

not found: /home/md2292/inception-frontier-open-weight-models/results/simple_inject/think/trial_0/DeepSeek-V4-Flash.pickle


## 3. Max-iter=1 ablation — 256 tokens (default)

In [4]:
df_abl256 = load('ablation_256')
if df_abl256 is not None:
    print(f'shape: {df_abl256.shape}')
    print(f'columns: {list(df_abl256.columns)}')
    display(df_abl256.head(3))

    arc_len  = df_abl256['architect_iteration_0'].str.len().dropna() if 'architect_iteration_0' in df_abl256.columns else pd.Series(dtype=float)
    tgt_len  = df_abl256['target_iteration_0'].str.len().dropna()    if 'target_iteration_0'   in df_abl256.columns else pd.Series(dtype=float)
    reas_len = df_abl256['reasoning'].str.len().dropna()              if 'reasoning'            in df_abl256.columns else pd.Series(dtype=float)

    print(f'  architect_iteration_0 mean len: {arc_len.mean():.0f}')
    print(f'  target_iteration_0    mean len: {tgt_len.mean():.0f}')
    print(f'  reasoning             mean len: {reas_len.mean():.0f}')

    fig, axes = plt.subplots(1, 3, figsize=(14, 3))
    for ax, data, label, color in zip(axes,
            [arc_len, tgt_len, reas_len],
            ['architect_iter_0', 'target_iter_0', 'reasoning'],
            ['slateblue', 'tomato', 'goldenrod']):
        ax.hist(data, bins=30, color=color)
        ax.set_title(f'{MODEL} abl256 — {label}'); ax.set_xlabel('chars')
    plt.tight_layout(); plt.show()

not found: /home/md2292/inception-frontier-open-weight-models/results/max_iterations_1/think/trial_0/DeepSeek-V4-Flash.pickle


## 4. Max-iter=1 ablations — all token budgets comparison

In [ ]:
ablation_variants = ['ablation_128', 'ablation_256', 'ablation_512', 'ablation_768', 'ablation_1024']
token_budgets     = [128, 256, 512, 768, 1024]

rows = []
for var, budget in zip(ablation_variants, token_budgets):
    df = load(var)
    if df is None:
        rows.append({'budget': budget, 'n_rows': 0, 'mean_arc_len': None, 'mean_tgt_len': None, 'mean_reas_len': None})
        continue
    rows.append({
        'budget':        budget,
        'n_rows':        len(df),
        'mean_arc_len':  df['architect_iteration_0'].str.len().mean() if 'architect_iteration_0' in df.columns else None,
        'mean_tgt_len':  df['target_iteration_0'].str.len().mean()    if 'target_iteration_0'   in df.columns else None,
        'mean_reas_len': df['reasoning'].str.len().mean()              if 'reasoning'            in df.columns else None,
    })

abl_summary = pd.DataFrame(rows).set_index('budget')
display(abl_summary)

available = abl_summary['mean_arc_len'].dropna()
if not available.empty:
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.bar(available.index.astype(str), available.values, color='mediumpurple', width=0.5)
    ax.set_xlabel('architect_initial_max_tokens'); ax.set_ylabel('mean architect_iter_0 length (chars)')
    ax.set_title(f'{MODEL} — architect output length by token budget')
    plt.tight_layout(); plt.show()

## 5. Max-iter=5 (full inception)

In [ ]:
df_m5 = load('max_iter_5')
if df_m5 is not None:
    print(f'shape: {df_m5.shape}')
    print(f'columns: {list(df_m5.columns)}')
    display(df_m5.head(3))

    # Identify how many iterations each row has
    max_iters = sum(1 for c in df_m5.columns if c.startswith('architect_iteration_'))
    print(f'iterations present: {max_iters}')

    # Per-iteration architect + target lengths
    iter_stats = []
    for i in range(max_iters):
        ac = f'architect_iteration_{i}'; tc = f'target_iteration_{i}'
        iter_stats.append({
            'iter': i,
            'mean_arc': df_m5[ac].str.len().mean() if ac in df_m5.columns else None,
            'mean_tgt': df_m5[tc].str.len().mean() if tc in df_m5.columns else None,
        })
    iter_df = pd.DataFrame(iter_stats).set_index('iter')
    display(iter_df)

    # Graduation: rows that have non-null target at the last iteration
    last_tgt = f'target_iteration_{max_iters-1}'
    if last_tgt in df_m5.columns:
        n_grad = df_m5[last_tgt].notna().sum()
        print(f'Rows with content at final iteration ({last_tgt}): {n_grad}/{len(df_m5)} ({100*n_grad/len(df_m5):.1f}%)')

    # Reasoning length distribution
    if 'reasoning' in df_m5.columns:
        plt.figure(figsize=(6, 3))
        plt.hist(df_m5['reasoning'].str.len().dropna(), bins=30, color='darkorange')
        plt.title(f'{MODEL} max_iter=5 — final reasoning length'); plt.xlabel('chars')
        plt.tight_layout(); plt.show()

## 6. Trial consistency (new-model trial data)

In [ ]:
# Toggle variant here — then re-run cell
TRIAL_VARIANT = 'ablation_128'   # or 'ablation_256', 'max_iter_5', 'simple_inject', 'benchmark'
N_TRIALS = 10

df_all = load_all_trials(TRIAL_VARIANT, model=MODEL, n_trials=N_TRIALS)
if df_all is not None:
    print(f'shape: {df_all.shape}, trials: {sorted(df_all["trial"].unique())}')

    # Response / reasoning length per trial
    len_col = 'reasoning' if 'reasoning' in df_all.columns else 'response'
    per_trial = df_all.groupby('trial')[len_col].apply(lambda s: s.str.len().mean())
    print(f'Mean {len_col} length per trial:')
    display(per_trial.to_frame('mean_len'))

    plt.figure(figsize=(8, 3))
    per_trial.plot(kind='bar', color='teal')
    plt.title(f'{MODEL} {TRIAL_VARIANT} — mean {len_col} len per trial')
    plt.xlabel('trial_idx'); plt.ylabel('mean chars'); plt.xticks(rotation=0)
    plt.tight_layout(); plt.show()

## 7. Raw row viewer

In [ ]:
# Toggle these to inspect any row in any experiment
VIEW_VARIANT = 'ablation_128'
VIEW_ROW = 0
TRUNCATE = 500   # chars per column; set None for full

df_view = load(VIEW_VARIANT)
if df_view is not None and VIEW_ROW < len(df_view):
    row = df_view.iloc[VIEW_ROW]
    for col in df_view.columns:
        val = str(row[col]) if row[col] is not None else 'None'
        val = val[:TRUNCATE] + '...' if TRUNCATE and len(val) > TRUNCATE else val
        print(f'\n── {col} ──\n{val}')